In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("house_price.csv")

In [5]:
df.shape

(18, 3)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   HouseSize   18 non-null     int64
 1   Bedrooms    18 non-null     int64
 2   HousePrice  18 non-null     int64
dtypes: int64(3)
memory usage: 564.0 bytes


In [7]:
df.head()

,HouseSize,Bedrooms,HousePrice
0,800,2,180000
1,900,2,200000
2,1000,2,220000
3,1100,2,240000
4,1200,3,260000


In [4]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [8]:
from sklearn.model_selection import train_test_split

In [10]:
##train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [21]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

### target scaling

In [34]:
scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
).ravel()

In [40]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

model = SVR(kernel="rbf")

param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.1, 0.01, 0.001],
    "epsilon": [0.1, 0.2, 0.5, 1]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train_scaled, y_train_scaled)

print("Best Parameters:", grid.best_params_)
print("Best CV R2:", grid.best_score_)

Best Parameters: {'C': 100, 'epsilon': 0.1, 'gamma': 0.01}
Best CV R2: 0.991282220133096


### Prediction

In [41]:
best_model = grid.best_estimator_

y_pred_scaled = best_model.predict(X_test_scaled)

y_pred = scaler_y.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).ravel()

### evaluation:

In [43]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("Test R2:", r2_score(y_test, y_pred))

MAE: 9175.353416218306
MSE: 98921343.19570096
RMSE: 9945.920932508008
Test R2: 0.9758728431229997
